# Atlas Support AI — Kaggle GPU training and reviewed evaluation

This notebook runs QLoRA supervised fine-tuning (SFT) and DPO on fictional synthetic examples, then compares both adapters on a separately maintained, manually reviewed fictional ticket set.

**Interpret results carefully:** all scenarios are fictional. The reviewed set is useful for a portfolio smoke evaluation, not evidence of real customer performance or production readiness. DPO scored substantially worse than SFT in the supplied run.

## Run order

Enable a Kaggle GPU and Internet if available, attach the `reviewed_triage.jsonl` dataset, then run the notebook in order. Training can take several minutes and downloads a base model. If the kernel is interrupted, training checkpoints and per-ticket evaluation progress can resume while the Kaggle session's `/kaggle/working` storage remains available. After completion, run the final packaging cell and save a Kaggle notebook version to retain outputs.

The notebook trains triage adapters only. It does not integrate them into the project's API or monitoring stack.


In [1]:
# Check Kaggle GPU availability and establish reproducible paths.
import os, random, json, gc, zipfile, shutil
from pathlib import Path
# This notebook trains on one GPU. Hide the second T4 before importing torch so
# Trainer does not activate unsupported nn.DataParallel automatically.
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
WORK = Path('/kaggle/working/atlas_support_ai')
WORK.mkdir(parents=True, exist_ok=True)
HF_HOME = WORK / 'hf-cache'
legacy_model_cache = (Path.home() / '.cache' / 'huggingface' / 'hub' /
                      'models--Qwen--Qwen2.5-3B-Instruct')
persistent_model_cache = HF_HOME / 'hub' / 'models--Qwen--Qwen2.5-3B-Instruct'
if legacy_model_cache.exists() and not persistent_model_cache.exists():
    persistent_model_cache.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(legacy_model_cache, persistent_model_cache, symlinks=True)
os.environ['HF_HOME'] = str(HF_HOME)
os.environ['HF_HUB_CACHE'] = str(HF_HOME / 'hub')
import torch

assert torch.cuda.is_available(), "Enable a GPU accelerator in Notebook Settings first."
print(torch.cuda.get_device_name(0))
print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GiB")
os.environ['MLFLOW_TRACKING_URI'] = f"sqlite:///{WORK / 'mlflow.db'}"
os.environ['MLFLOW_EXPERIMENT_NAME'] = 'atlas-triage-synthetic-smoke'
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)


Tesla T4
GPU memory: 14.6 GiB


In [2]:
# Install a known-compatible training stack. This may take several minutes.
# Use modern wheels for Kaggle's current Python runtime. Older Transformers pins
# require tokenizers releases that may fall back to a Rust source build.
%pip install -q --upgrade pip
%pip install -q --only-binary=:all: --upgrade pyarrow
%pip install -q transformers==5.13.1 trl==1.17.1 peft==0.21.2 datasets==5.1.0 accelerate==1.15.0 bitsandbytes==0.50.2 mlflow-skinny sqlalchemy alembic sqlparse


Note: you may need to restart the kernel to use updated packages.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
# Create 700 synthetic train tickets and 70 synthetic holdout tickets.
from datasets import Dataset

cases = {
 'bug': ('bug','normal','platform',['The {component} returns {symptom} after {trigger}.','Since {trigger}, our {component} has {symptom}.']),
 'billing': ('billing','high','billing',['I see a duplicate charge for {item}.','Please explain the unexpected {item} charge on my account.']),
 'account': ('account','high','identity',['I cannot sign in after {trigger}.','My account access stopped working after {trigger}.']),
 'feature_request': ('feature_request','low','general',['Please add {feature} to the product.','Our team would like a way to {feature}.']),
 'how_to': ('how_to','low','general',['How can I {action} in the workspace?','Where do I find the steps to {action}?']),
 'security': ('security','urgent','security',['I found a possible {incident} in our workspace.','Please investigate this suspected {incident} immediately.']),
 'other': ('other','normal','general',['I have a question about {topic}.','Can someone help clarify {topic}?'])}
values = {
 'component':['API','dashboard','export job','integration','mobile app'],
 'symptom':['an error','a timeout','an empty response','unexpected results','a blank screen'],
 'trigger':['the latest update','a settings change','yesterday','a new deployment','sign-in'],
 'item':['the monthly plan','an add-on','an invoice','a renewal','a seat'],
 'feature':['scheduled exports','dark mode','bulk editing','custom alerts','a calendar view'],
 'action':['rotate an API credential','invite a teammate','export a report','change a role','configure alerts'],
 'incident':['exposed credential','unexpected login','permission issue','suspicious access','data leak'],
 'topic':['workspace settings','an integration','our subscription','user roles','data exports']}
labels = list(cases)

def make_example(i):
    name = labels[i % len(labels)]
    category, priority, area, templates = cases[name]
    vals = {key: values[key][(i+j) % len(values[key])] for j,key in enumerate(values)}
    ticket = templates[(i // len(labels)) % len(templates)].format(**vals)
    target = {'category':category,'priority':priority,'product_area':area,'summary':' '.join(ticket.split())[:180]}
    prompt = ('Classify the fictional support ticket. Return only JSON with category, priority, '
              f'product_area and summary.\nTicket: {ticket}\nJSON:')
    return ticket, target, prompt

train = [make_example(i) for i in range(700)]
holdout = [make_example(i+700) for i in range(70)]
sft_rows, dpo_rows = [], []
for ticket, target, prompt in train:
    sft_rows.append({'text': prompt + ' ' + json.dumps(target, ensure_ascii=False) + '\n'})
    wrong = dict(target)
    wrong['category'] = labels[(labels.index(target['category']) + 1) % len(labels)]
    dpo_rows.append({'prompt':prompt, 'chosen':json.dumps(target, ensure_ascii=False),
                     'rejected':json.dumps(wrong, ensure_ascii=False)})
print(f"SFT rows: {len(sft_rows)} | DPO preference pairs: {len(dpo_rows)} | holdout: {len(holdout)}")
print('All records are synthetic and must not be described as human-reviewed.')


SFT rows: 700 | DPO preference pairs: 700 | holdout: 70
All records are synthetic and must not be described as human-reviewed.


In [4]:
# Load the licensed base model in 4-bit and run QLoRA supervised fine-tuning.
import mlflow
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, Trainer
from peft import LoraConfig, prepare_model_for_kbit_training, PeftModel
from trl import SFTConfig, SFTTrainer, DPOConfig, DPOTrainer

# Transformers 5.x calculates a diagnostic gradient norm even when clipping is
# disabled. On Kaggle's T4/FP16 stack that diagnostic raises NotImplementedError;
# override only the diagnostic, while max_grad_norm=0 keeps clipping disabled.
def skip_grad_norm(self, model, grad_norm=None):
    return 0.0 if grad_norm is None else grad_norm

Trainer._get_grad_norm = skip_grad_norm

BASE_MODEL = 'Qwen/Qwen2.5-3B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=torch.float16,
                         bnb_4bit_use_double_quant=True)
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                  target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
                  task_type='CAUSAL_LM')

def load_base():
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb,
                                                  device_map={'':0}, dtype=torch.float16)
    model.config.use_cache = False
    return prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

def force_trainable_fp32(model):
    # Prevent BF16 LoRA gradients from reaching the FP16 GradScaler path on T4.
    for param in model.parameters():
        if param.requires_grad and param.dtype != torch.float32:
            param.data = param.data.to(torch.float32)

mlflow.set_tracking_uri(os.environ['MLFLOW_TRACKING_URI'])
mlflow.set_experiment(os.environ['MLFLOW_EXPERIMENT_NAME'])
sft_dir = WORK / 'triage-sft-adapter'
sft_args = SFTConfig(output_dir=str(sft_dir), num_train_epochs=1,
    per_device_train_batch_size=1, gradient_accumulation_steps=8,
    learning_rate=2e-4, logging_steps=20, save_strategy='steps',
    save_steps=25, save_total_limit=2,
    max_grad_norm=0.0,
    fp16=False, bf16=False, gradient_checkpointing=True,
    optim='adamw_torch', report_to=[], seed=SEED,
    remove_unused_columns=False, dataset_text_field='text', max_length=1024)
sft_trainer = SFTTrainer(model=load_base(), args=sft_args,
    train_dataset=Dataset.from_list(sft_rows), peft_config=lora,
    processing_class=tokenizer)
force_trainable_fp32(sft_trainer.model)
with mlflow.start_run(run_name='qlora-sft-synthetic-smoke'):
    mlflow.log_params({'base_model':BASE_MODEL,'train_rows':len(sft_rows),'epochs':1,
                       'method':'QLoRA SFT','data':'synthetic smoke test'})
    from transformers.trainer_utils import get_last_checkpoint
    sft_checkpoint = get_last_checkpoint(str(sft_dir))
    sft_result = sft_trainer.train(resume_from_checkpoint=sft_checkpoint)
    sft_trainer.save_model(str(sft_dir))
    mlflow.log_metrics({k:float(v) for k,v in sft_result.metrics.items() if isinstance(v,(int,float))})
    mlflow.log_artifacts(str(sft_dir), artifact_path='adapter')
print('SFT adapter saved:', sft_dir)


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

2026/11/07 16:11:03 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/07 16:11:03 INFO mlflow.store.db.utils: Updating database tables
2026/11/07 16:11:05 INFO mlflow.tracking.fluent: Experiment with name 'atlas-triage-synthetic-smoke' does not exist. Creating a new experiment.


... (9 outputs left out)

Dropping fully masked examples from train dataset:   0%|          | 0/700 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss
20,1.135130
40,0.212563
60,0.103479
80,0.087535


SFT adapter saved: /kaggle/working/atlas_support_ai/triage-sft-adapter


In [5]:
# Optional SFT recovery cell. Run this manually only if the SFT cell above failed.
# It reuses the already-loaded model and resumes the latest saved checkpoint.
if 'sft_trainer' not in globals():
    print('No SFT trainer is loaded. Run the model-loading/SFT cell first.')
elif 'sft_result' in globals():
    print('SFT already completed; recovery skipped.')
else:
    from transformers import Trainer
    from transformers.trainer_utils import get_last_checkpoint

    # Skip the unsupported gradient-norm diagnostic in this Kaggle session.
    def skip_grad_norm(self, model, grad_norm=None):
        return 0.0 if grad_norm is None else grad_norm

    Trainer._get_grad_norm = skip_grad_norm
    sft_trainer.args.max_grad_norm = 0.0
    sft_trainer.args.fp16 = False
    sft_trainer.args.bf16 = False
    sft_trainer.accelerator.scaler = None
    sft_trainer.accelerator.native_amp = False
    force_trainable_fp32(sft_trainer.model)
    sft_trainer.args.optim = 'adamw_torch'
    sft_trainer.optimizer = None
    sft_trainer.lr_scheduler = None
    sft_trainer.model.zero_grad(set_to_none=True)

    checkpoint = get_last_checkpoint(str(sft_dir))
    with mlflow.start_run(run_name='qlora-sft-synthetic-smoke-retry'):
        sft_result = sft_trainer.train(resume_from_checkpoint=checkpoint)
        sft_trainer.save_model(str(sft_dir))
        mlflow.log_metrics({
            k: float(v) for k, v in sft_result.metrics.items()
            if isinstance(v, (int, float))
        })
        mlflow.log_artifacts(str(sft_dir), artifact_path='adapter')

    print('SFT finished:', sft_dir)


SFT already completed; recovery skipped.


In [10]:
# Run DPO over 700 synthetic preference pairs. This is smoke training only.
# These synthetic pairs are for demonstration, not human preference alignment.
if "sft_trainer" in globals():
    del sft_trainer
gc.collect()
torch.cuda.empty_cache()

dpo_base = load_base()
dpo_model = PeftModel.from_pretrained(
    dpo_base,
    str(sft_dir),
    is_trainable=True,
)
dpo_model.config.use_cache = False

dpo_dir = WORK / "triage-dpo-adapter"
dpo_args = DPOConfig(
    output_dir=str(dpo_dir),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    learning_rate=5e-5,
    logging_steps=20,
    save_strategy="steps",
    save_steps=25,
    save_total_limit=2,
    max_grad_norm=0.0,
    fp16=False,
    bf16=False,
    gradient_checkpointing=True,
    optim="adamw_torch",
    report_to=[],
    remove_unused_columns=False,
    seed=SEED,
    beta=0.1,
    max_length=512,
)

dpo_trainer = DPOTrainer(
    model=dpo_model,
    args=dpo_args,
    train_dataset=Dataset.from_list(dpo_rows),
    processing_class=tokenizer,
)
force_trainable_fp32(dpo_trainer.model)

from transformers.trainer_utils import get_last_checkpoint

dpo_checkpoint = get_last_checkpoint(str(dpo_dir))

with mlflow.start_run(run_name="dpo-synthetic-smoke"):
    mlflow.log_params({
        "base_model": BASE_MODEL,
        "preference_pairs": len(dpo_rows),
        "epochs": 1,
        "beta": 0.1,
        "method": "DPO",
        "data": "synthetic smoke test",
    })

    dpo_result = dpo_trainer.train(
        resume_from_checkpoint=dpo_checkpoint
    )
    dpo_trainer.save_model(str(dpo_dir))

    mlflow.log_metrics({
        k: float(v)
        for k, v in dpo_result.metrics.items()
        if isinstance(v, (int, float))
    })
    mlflow.log_artifacts(str(dpo_dir), artifact_path="adapter")

print("DPO adapter saved:", dpo_dir)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Adding EOS to train dataset:   0%|          | 0/700 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/700 [00:00<?, ? examples/s]

Dropping fully truncated examples from train dataset:   0%|          | 0/700 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss
20,0.073427
40,0.000375
60,0.000155
80,0.000098


DPO adapter saved: /kaggle/working/atlas_support_ai/triage-dpo-adapter


In [ ]:
# Load and validate the separately attached reviewed fictional evaluation set.
import json
from pathlib import Path

matches = sorted(Path("/kaggle/input").rglob("reviewed_triage.jsonl"))
assert matches, "Attach the reviewed_triage.jsonl dataset using Add Data."
eval_path = matches[0]
rows = [
    json.loads(line)
    for line in eval_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
required = {"category", "priority", "product_area"}
for i, row in enumerate(rows, start=1):
    assert row.get("ticket"), f"Row {i} has no ticket text."
    assert required <= set(row.get("label", {})), f"Row {i} is missing label fields."
records = [{"ticket": row["ticket"], "gold": row["label"]} for row in rows]
assert len(records) == 70, f"Expected the reviewed 70-ticket set, found {len(records)}."
print(f"Loaded {len(records)} reviewed fictional tickets from {eval_path}")


In [ ]:
# Release DPO training memory and load the trained SFT adapter for evaluation.
import gc
import torch
from peft import PeftModel

for name in ("dpo_trainer", "dpo_model", "dpo_base"):
    if name in globals():
        del globals()[name]
gc.collect()
torch.cuda.empty_cache()
sft_base = load_base()
sft_model = PeftModel.from_pretrained(sft_base, str(sft_dir), is_trainable=False)
sft_model.eval()
print("SFT adapter ready:", sft_dir)


In [ ]:
# Resume-safe SFT evaluation: checkpoints one prediction at a time.
import json, re, time
from pathlib import Path
import torch

assert "records" in globals() and records, "Run the reviewed dataset loader first."
assert "sft_model" in globals(), "Load the SFT adapter first."
base = Path("/kaggle/working/atlas_support_ai")
base.mkdir(parents=True, exist_ok=True)
progress_path = base / "sft_reviewed_eval_progress.jsonl"
metrics_path = base / "sft_reviewed_eval_metrics.json"
predictions_path = base / "sft_reviewed_eval_predictions.jsonl"

done = {}
if progress_path.exists():
    for line in progress_path.read_text(encoding="utf-8").splitlines():
        try:
            item = json.loads(line)
            i = int(item["index"])
            if 0 <= i < len(records) and item.get("ticket") == records[i]["ticket"]:
                done[i] = item["prediction"]
        except (ValueError, KeyError, TypeError, json.JSONDecodeError):
            continue
print(f"SFT evaluation starting: {len(done)}/{len(records)} already checkpointed", flush=True)
sft_model.eval()
started = time.time()
for i, row in enumerate(records):
    if i in done:
        continue
    prompt = (
        "Classify the fictional support ticket. Return only JSON with category, "
        "priority, product_area and summary.\n"
        f"Ticket: {row['ticket']}\nJSON:"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda:0")
    with torch.no_grad():
        output = sft_model.generate(
            **inputs, max_new_tokens=120, do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    raw = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    match = re.search(r"\{.*?\}", raw, re.S)
    try:
        prediction = json.loads(match.group(0)) if match else {}
    except json.JSONDecodeError:
        prediction = {}
    item = {"index": i, "ticket": row["ticket"], "gold": row["gold"], "prediction": prediction}
    with progress_path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")
        f.flush()
    done[i] = prediction
    if len(done) % 5 == 0 or len(done) == len(records):
        print(f"SFT evaluated {len(done)}/{len(records)} ({(time.time()-started)/60:.1f} min this run)", flush=True)

def macro_f1(gold, pred):
    scores = []
    for label in sorted(set(gold) | set(pred)):
        tp = sum(g == label and p == label for g, p in zip(gold, pred))
        fp = sum(g != label and p == label for g, p in zip(gold, pred))
        fn = sum(g == label and p != label for g, p in zip(gold, pred))
        precision, recall = tp / max(1, tp + fp), tp / max(1, tp + fn)
        scores.append(2 * precision * recall / max(1e-12, precision + recall))
    return sum(scores) / max(1, len(scores))

ordered = [done[i] for i in range(len(records))]
metrics = {"examples": len(records), "json_validity": sum(
    all(k in p for k in ("category", "priority", "product_area")) for p in ordered
) / len(records)}
for field in ("category", "priority", "product_area"):
    gold = [r["gold"][field] for r in records]
    pred = [p.get(field, "__invalid__") for p in ordered]
    metrics[f"{field}_accuracy"] = sum(a == b for a, b in zip(gold, pred)) / len(records)
    if field == "category":
        metrics["category_macro_f1"] = macro_f1(gold, pred)
metrics["data_status"] = "fictional manually reviewed scenarios; not real-customer acceptance evidence"
metrics_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
with predictions_path.open("w", encoding="utf-8") as f:
    for i, row in enumerate(records):
        f.write(json.dumps({"index": i, "ticket": row["ticket"], "gold": row["gold"], "prediction": ordered[i]}, ensure_ascii=False) + "\n")
print("SFT evaluation complete:")
print(json.dumps(metrics, indent=2))
print("Checkpoint:", progress_path)
print("Predictions:", predictions_path)


In [ ]:
import gc
import torch
from peft import PeftModel

del sft_model
gc.collect()
torch.cuda.empty_cache()

dpo_base = load_base()
dpo_model = PeftModel.from_pretrained(
    dpo_base, str(dpo_dir), is_trainable=False
)
dpo_model.eval()

In [ ]:
# Resumable DPO evaluation on the uploaded reviewed ticket set.
# Each completed prediction is appended immediately; rerunning skips matching tickets.
import json, re, time
from pathlib import Path
import torch

assert "records" in globals() and len(records) > 0, "Run the reviewed dataset-loading cell first."
assert "dpo_model" in globals(), "Run the DPO adapter-loading cell first."

progress_path = Path("/kaggle/working/atlas_support_ai/dpo_reviewed_eval_progress.jsonl")
metrics_path = Path("/kaggle/working/atlas_support_ai/dpo_reviewed_eval_metrics.json")
predictions_path = Path("/kaggle/working/atlas_support_ai/dpo_reviewed_eval_predictions.jsonl")
progress_path.parent.mkdir(parents=True, exist_ok=True)

# Recover only checkpoint rows whose index and ticket still match this evaluation set.
done = {}
if progress_path.exists():
    for line in progress_path.read_text(encoding="utf-8").splitlines():
        try:
            item = json.loads(line)
            i = int(item["index"])
            if 0 <= i < len(records) and item["ticket"] == records[i]["ticket"]:
                done[i] = item["prediction"]
        except (ValueError, KeyError, TypeError, json.JSONDecodeError):
            continue

print(f"DPO evaluation starting: {len(done)}/{len(records)} already checkpointed", flush=True)
dpo_model.eval()
started = time.time()
for i, row in enumerate(records):
    if i in done:
        continue
    prompt = (
        "Classify the fictional support ticket. Return only JSON with category, "
        "priority, product_area and summary.\n"
        f"Ticket: {row['ticket']}\nJSON:"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda:0")
    with torch.no_grad():
        output = dpo_model.generate(
            **inputs, max_new_tokens=120, do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    raw = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )
    match = re.search(r"\{.*?\}", raw, re.S)
    try:
        prediction = json.loads(match.group(0)) if match else {}
    except json.JSONDecodeError:
        prediction = {}
    item = {"index": i, "ticket": row["ticket"], "gold": row["gold"],
            "prediction": prediction, "raw_output": raw}
    with progress_path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")
        f.flush()
    done[i] = prediction
    if len(done) % 5 == 0 or len(done) == len(records):
        print(f"DPO evaluated {len(done)}/{len(records)} ({(time.time()-started)/60:.1f} min this run)", flush=True)

# Compute field accuracy and category macro-F1 from all checkpointed predictions.
def macro_f1(gold, pred):
    scores = []
    for label in sorted(set(gold) | set(pred)):
        tp = sum(g == label and p == label for g, p in zip(gold, pred))
        fp = sum(g != label and p == label for g, p in zip(gold, pred))
        fn = sum(g == label and p != label for g, p in zip(gold, pred))
        precision = tp / max(1, tp + fp)
        recall = tp / max(1, tp + fn)
        scores.append(2 * precision * recall / max(1e-12, precision + recall))
    return sum(scores) / max(1, len(scores))

ordered = [done[i] for i in range(len(records))]
metrics = {"examples": len(records), "json_validity": sum(
    all(k in p for k in ("category", "priority", "product_area")) for p in ordered
) / max(1, len(ordered))}
for field in ("category", "priority", "product_area"):
    gold = [r["gold"].get(field) for r in records]
    pred = [p.get(field, "__invalid__") for p in ordered]
    metrics[f"{field}_accuracy"] = sum(a == b for a, b in zip(gold, pred)) / max(1, len(gold))
    if field == "category":
        metrics["category_macro_f1"] = macro_f1(gold, pred)
metrics["data_status"] = "fictional manually reviewed scenarios; not real-customer acceptance evidence"
metrics_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
with predictions_path.open("w", encoding="utf-8") as f:
    for i, row in enumerate(records):
        f.write(json.dumps({"index": i, "ticket": row["ticket"], "gold": row["gold"],
                            "prediction": ordered[i]}, ensure_ascii=False) + "\n")
print("DPO evaluation complete:")
print(json.dumps(metrics, indent=2))
print("Per-ticket checkpoint:", progress_path)
print("Predictions:", predictions_path)
print("Metrics:", metrics_path)


In [ ]:
# Compare the two adapters on exactly the same reviewed fictional tickets.
import json
from pathlib import Path

base = Path("/kaggle/working/atlas_support_ai")
sft_path = base / "sft_reviewed_eval_predictions.jsonl"
dpo_path = base / "dpo_reviewed_eval_predictions.jsonl"
assert sft_path.exists() and dpo_path.exists(), "Run both reviewed evaluation cells first."
def load_indexed(path):
    rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    return {int(row["index"]): row for row in rows}
sft, dpo = load_indexed(sft_path), load_indexed(dpo_path)
assert set(sft) == set(dpo) == set(range(len(records))), "Evaluation files do not contain the same complete ticket set."
for i in range(len(records)):
    assert sft[i]["ticket"] == dpo[i]["ticket"] == records[i]["ticket"], f"Ticket mismatch at index {i}."
    assert sft[i]["gold"] == dpo[i]["gold"] == records[i]["gold"], f"Gold-label mismatch at index {i}."
comparison = {"examples": len(records), "data_status": "fictional manually reviewed scenarios; not real-customer acceptance evidence", "fields": {}}
for field in ("category", "priority", "product_area"):
    sft_ok = sum(sft[i]["prediction"].get(field) == sft[i]["gold"].get(field) for i in sft)
    dpo_ok = sum(dpo[i]["prediction"].get(field) == dpo[i]["gold"].get(field) for i in dpo)
    comparison["fields"][field] = {"sft_correct": sft_ok, "dpo_correct": dpo_ok, "total": len(records)}
    print(f"{field}: SFT {sft_ok}/{len(records)} ({sft_ok/len(records):.1%}) | DPO {dpo_ok}/{len(records)} ({dpo_ok/len(records):.1%})")
(base / "reviewed_eval_comparison.json").write_text(json.dumps(comparison, indent=2), encoding="utf-8")
print("Compared tickets:", len(records))
print("Saved:", base / "reviewed_eval_comparison.json")


In [ ]:
# Build a compact final artifact bundle without rerunning training or inference.
import json, zipfile
from pathlib import Path

base = Path("/kaggle/working/atlas_support_ai")
required = [
    base / "sft_reviewed_eval_metrics.json",
    base / "sft_reviewed_eval_predictions.jsonl",
    base / "dpo_reviewed_eval_metrics.json",
    base / "dpo_reviewed_eval_predictions.jsonl",
    base / "reviewed_eval_comparison.json",
]
missing = [str(p) for p in required if not p.is_file()]
assert not missing, "Run both evaluation cells and the comparison cell first; missing: " + ", ".join(missing)
bundle = Path("/kaggle/working/atlas_support_ai_reviewed_bundle.zip")
with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as z:
    # Include only final adapter files; omit large duplicate training checkpoints and DPO reference copies.
    for name, folder in (("triage-sft-adapter", sft_dir), ("triage-dpo-adapter", dpo_dir)):
        assert folder.is_dir(), f"Adapter directory not found: {folder}"
        for file in folder.iterdir():
            if file.is_file():
                z.write(file, f"{name}/{file.name}")
    for file in required:
        z.write(file, file.name)
print("Final reviewed-evaluation bundle:", bundle)
print("Included: final SFT/DPO adapters, reviewed metrics, per-ticket predictions, and comparison.")


## Results and final artifact

After both adapter evaluations finish, run the comparison cell and then the final packaging cell. The generated `atlas_support_ai_reviewed_bundle.zip` contains the final SFT and DPO adapter files, the reviewed evaluation metrics, per-ticket predictions, and the side-by-side comparison. Training checkpoints and the DPO reference adapter are omitted to keep the download smaller.

The supplied run produced these results on 70 fictional manually reviewed scenarios:

| Field | SFT | DPO |
|---|---:|---:|
| Category accuracy | 62/70 (88.6%) | 17/70 (24.3%) |
| Priority accuracy | 63/70 (90.0%) | 24/70 (34.3%) |
| Product-area accuracy | 65/70 (92.9%) | 9/70 (12.9%) |

DPO JSON validity was 67.1%; its category macro-F1 was 0.139. In this run SFT was the stronger candidate. These figures are a fictional portfolio smoke evaluation, not real-customer quality or production-acceptance evidence. The adapters are not automatically wired into the project's API or vLLM deployment.
